In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os

# Load Data

### DXA Sleep actigraphy

In [10]:
folder_path = '/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/DXA_sleep_actigraphy'


In [11]:
def find_header_row(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        non_blank_index = 0
        for i, line in enumerate(f):
            if line.strip() == '':
                continue
            if (
                'Line'             in line and
                'Date'             in line and
                'Time'             in line and
                'Activity'         in line and
                'Interval Status'  in line and
                'Sleep/Wake'       in line
            ):
                return non_blank_index
            non_blank_index += 1
    raise ValueError(f'Header row not found in {filepath}')

In [12]:
# ## Step 1 define header path

# def find_header_row(filepath):
#     with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
#         non_blank_index = 0
#         for i, line in enumerate(f):
#             if line.strip() == '':  # skip blank lines, don't increment counter
#                 continue
#             if (
#                 '"Line"' in line
#                 and '"Date"' in line
#                 and '"Time"' in line
#                 and '"Activity"' in line
#                 and '"Interval Status"' in line
#             ):
#                 return non_blank_index
#             non_blank_index += 1

#     raise ValueError(f'Header row not found in {filepath}')

In [13]:
## Step 3 Collect file metadata
files_to_import = []

for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        filepath = os.path.join(folder_path, filename)
        files_to_import.append({
            'file': filename,
            'name': os.path.splitext(filename)[0].lower(),
            'filepath': filepath,
            'header_row': find_header_row(filepath),
            'delimiter': ',',
            'usecols': list(range(12))
        })


In [14]:
## Check/confirm header row locations

for f in files_to_import:
    print(f['name'], '→ header row:', f['header_row'])

dxa174_3_27_2025_12_49_00_pm_174_24hr_ah → header row: 293
dxa022_10_31_2019_1_56_00_pm_cc_24hr_combined → header row: 264
dxa025_11_5_2019_2_34_00_pm_cc_24hr_combined → header row: 197
dxa110_10_31_2023_1_23_00_pm_cc_24hr_combined → header row: 198
dxa002_9_24_2019_10_04_00_am_cc_24hr_combined → header row: 188
dxa086_9_19_2023_10_36_00_am_cc_24hr_combined → header row: 181
dxa142_4_5_2024_1_03_00_pm_cc_24hr_combined → header row: 235
dxa170_2_21_2025_2_05_00_pm_cc_24hr_combined → header row: 232
dxa046_2_7_2020_2_23_00_pm_dxa046_1_24hr_ah → header row: 122
dxa178_6_2_2025_1_18_00_pm_dxa178_24hr_ah → header row: 203
dxa086_10_4_2023_2_26_00_pm_cc_24hr_combined → header row: 138
dxa056_2_8_2023_12_53_00_pm_cc_24hr_combined → header row: 210
dxa023_11_1_2019_9_38_00_am_cc_24hr_combined → header row: 215
dxa040_1_24_2020_10_59_00_am_cc_24hr_combined → header row: 215
dxa070_3_22_2023_12_31_00_pm_cc_24hr_combined → header row: 222
dxa156_9_27_2024_3_00_00_pm_cc_24hr_combined → header row:

In [15]:
## Step 4 Import each file into a DataFrame
dataframes = {}
failed_files = []

for f in files_to_import:
    try:
        df = pd.read_csv(
            f['filepath'],
            header=f['header_row'],
            sep=f['delimiter'],
            usecols=f['usecols'],
            encoding='utf-8',
            encoding_errors='ignore',
            skip_blank_lines=True       # must match how find_header_row counts
        )

        ## Strip quotes from column names e.g. "Line" → Line
        df.columns = df.columns.str.strip('"').str.strip()

        ## Drop any fully blank rows
        df.dropna(how='all', inplace=True)

        ## Reset index so it runs 0, 1, 2... cleanly
        df.reset_index(drop=True, inplace=True)

        dataframes[f['name']] = df

    except Exception as e:
        print(f"FAILED: {f['file']} — {e}")
        failed_files.append(f['file'])

In [16]:
## Verify things uploaded properly
print(f"\n{len(dataframes)} files loaded successfully")
print(f"{len(failed_files)} files failed: {failed_files}")

## Spot-check one file
sample_key = list(dataframes.keys())[0]
print(f"\nSample: {sample_key}")
print(dataframes[sample_key].columns.tolist())
print(dataframes[sample_key].head())


80 files loaded successfully
0 files failed: []

Sample: dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
['Line', 'Date', 'Time', 'Off-Wrist Status', 'Activity', 'Marker', 'White Light', 'Red Light', 'Green Light', 'Blue Light', 'Sleep/Wake', 'Interval Status']
   Line       Date         Time  Off-Wrist Status  Activity  Marker  \
0     1  3/27/2025  12:49:00 PM                 0       NaN     0.0   
1     2  3/27/2025  12:50:00 PM                 0       NaN     0.0   
2     3  3/27/2025  12:51:00 PM                 0       NaN     0.0   
3     4  3/27/2025  12:52:00 PM                 0       NaN     0.0   
4     5  3/27/2025  12:53:00 PM                 0       NaN     0.0   

   White Light  Red Light  Green Light  Blue Light  Sleep/Wake Interval Status  
0          NaN        NaN          NaN         NaN         NaN          ACTIVE  
1          NaN        NaN          NaN         NaN         NaN          ACTIVE  
2          NaN        NaN          NaN         NaN         NaN          AC

In [17]:
# dataframes['dxa174_3_27_2025_12_49_00_pm_174_24hr_ah']

## sample check

In [18]:
## Concatenate all dataframes
DXA_actigraphy_combined_df = pd.concat(
    [df.assign(participant=name) for name, df in dataframes.items()],
    ignore_index=True
)

# Verify
print(DXA_actigraphy_combined_df.shape)
print(DXA_actigraphy_combined_df['participant'].nunique(), 'participants')
print(DXA_actigraphy_combined_df.head())

(1798329, 13)
80 participants
   Line       Date         Time  Off-Wrist Status  Activity  Marker  \
0   1.0  3/27/2025  12:49:00 PM               0.0       NaN     0.0   
1   2.0  3/27/2025  12:50:00 PM               0.0       NaN     0.0   
2   3.0  3/27/2025  12:51:00 PM               0.0       NaN     0.0   
3   4.0  3/27/2025  12:52:00 PM               0.0       NaN     0.0   
4   5.0  3/27/2025  12:53:00 PM               0.0       NaN     0.0   

   White Light  Red Light  Green Light  Blue Light  Sleep/Wake  \
0          NaN        NaN          NaN         NaN         NaN   
1          NaN        NaN          NaN         NaN         NaN   
2          NaN        NaN          NaN         NaN         NaN   
3          NaN        NaN          NaN         NaN         NaN   
4          NaN        NaN          NaN         NaN         NaN   

  Interval Status                               participant  
0          ACTIVE  dxa174_3_27_2025_12_49_00_pm_174_24hr_ah  
1          ACTIVE  dxa

In [19]:
DXA_actigraphy_combined_df.columns

Index(['Line', 'Date', 'Time', 'Off-Wrist Status', 'Activity', 'Marker',
       'White Light', 'Red Light', 'Green Light', 'Blue Light', 'Sleep/Wake',
       'Interval Status', 'participant'],
      dtype='object')

In [20]:
## Cleaning participant ID column names
DXA_actigraphy_combined_df['subject_id'] = DXA_actigraphy_combined_df['participant'].str[0:6]
DXA_actigraphy_combined_df['subject_id'] = DXA_actigraphy_combined_df['subject_id'].astype(str).str.extract(r'(\d+)')
DXA_actigraphy_combined_df['subject_id'] = DXA_actigraphy_combined_df['subject_id'].astype(int).apply(lambda x: f'DXA_{x:03d}')
DXA_actigraphy_combined_df['study'] = 'DXA'

DXA_actigraphy_combined_df.head()

,Line,Date,Time,Off-Wrist Status,Activity,Marker,White Light,Red Light,Green Light,Blue Light,Sleep/Wake,Interval Status,participant,subject_id,study
0,1.0,3/27/2025,12:49:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
1,2.0,3/27/2025,12:50:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
2,3.0,3/27/2025,12:51:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
3,4.0,3/27/2025,12:52:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
4,5.0,3/27/2025,12:53:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA


### Siesta 2 actigraphy

In [21]:
siesta_2_folder_path = '/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/Siesta_2_actigraphy'

In [22]:
## Step 3 Collect file metadata
siesta2_files_to_import = []

for filename in os.listdir(siesta_2_folder_path):
    if filename.endswith('.csv'):
        filepath = os.path.join(siesta_2_folder_path, filename)
        siesta2_files_to_import.append({
            'file': filename,
            'name': os.path.splitext(filename)[0].lower(),
            'filepath': filepath,
            'header_row': find_header_row(filepath),
            'delimiter': ',',
            'usecols': list(range(12))
        })


In [23]:
## Check/confirm header row locations
for f in siesta2_files_to_import:
    print(f['name'], '→ header row:', f['header_row'])

si275_2_7_2025_8_20_00_pm_si275_24hr_ah → header row: 286
136_1_28_2020_6_00_00_pm_si136_24hr_ah → header row: 187
124_1_28_2020_6_00_00_pm_si124_24hr_ah → header row: 201
133_1_28_2020_6_00_00_pm_si133_24hr_ah → header row: 218
si317_3_26_2025_6_49_00_pm_si317_24hr_ah → header row: 239
si249_10_28_2024_7_03_00_pm_si249_24hr_ah → header row: 182
si233_10_23_2024_7_55_00_pm_si233_24hr_ah → header row: 167
si411_2_5_2026_8_09_00_pm_si411_24hr_ah → header row: 193
si293_3_26_2025_7_32_00_pm_si293_24hr_ah → header row: 193
si423_2_5_2026_10_22_00_pm_si423_24hrl_ah → header row: 200
si386_10_21_2025_7_09_00_pm_si386_24hr_ah → header row: 201
174_1_30_2020_6_00_00_pm_si174_24hr_ah → header row: 154
si305_2_14_2025_6_43_00_pm_si305_24hr_ah → header row: 210
si218_10_21_2024_8_28_00_pm_si218_24hr_ah → header row: 175
si230_10_23_2024_10_18_00_am_si230_24hr_ah → header row: 197
si318_3_26_2025_7_06_00_pm_si318_24hr_ah → header row: 222
si269_2_5_2025_11_01_00_pm_si269_24hr_ah → header row: 208


In [24]:
## Step 4 Import each file into a DataFrame
siesta2_dataframes = {}
siesta2_failed_files = []

for f in siesta2_files_to_import:
    try:
        df = pd.read_csv(
            f['filepath'],
            header=f['header_row'],
            sep=f['delimiter'],
            usecols=f['usecols'],
            encoding='utf-8',
            encoding_errors='ignore',
            skip_blank_lines=True       # must match how find_header_row counts
        )

        ## Strip quotes from column names e.g. "Line" → Line
        df.columns = df.columns.str.strip('"').str.strip()

        ## Drop any fully blank rows
        df.dropna(how='all', inplace=True)

        ## Reset index so it runs 0, 1, 2... cleanly
        df.reset_index(drop=True, inplace=True)

        siesta2_dataframes[f['name']] = df

    except Exception as e:
        print(f"FAILED: {f['file']} — {e}")
        siesta2_failed_files.append(f['file'])

In [25]:
## Verify things uploaded properly
print(f"\n{len(siesta2_dataframes)} files loaded successfully")
print(f"{len(siesta2_failed_files)} files failed: {failed_files}")

## Spot-check one file
sample_key = list(siesta2_dataframes.keys())[0]
print(f"\nSample: {sample_key}")
print(siesta2_dataframes[sample_key].columns.tolist())
print(siesta2_dataframes[sample_key].head())


34 files loaded successfully
0 files failed: []

Sample: si275_2_7_2025_8_20_00_pm_si275_24hr_ah
['Line', 'Date', 'Time', 'Off-Wrist Status', 'Activity', 'Marker', 'White Light', 'Red Light', 'Green Light', 'Blue Light', 'Sleep/Wake', 'Interval Status']
   Line      Date        Time  Off-Wrist Status  Activity  Marker  \
0     1  2/7/2025  8:20:00 PM                 0       NaN     0.0   
1     2  2/7/2025  8:21:00 PM                 0     452.0     0.0   
2     3  2/7/2025  8:22:00 PM                 0     706.0     0.0   
3     4  2/7/2025  8:23:00 PM                 0     801.0     0.0   
4     5  2/7/2025  8:24:00 PM                 0     801.0     0.0   

   White Light  Red Light  Green Light  Blue Light  Sleep/Wake Interval Status  
0          NaN        NaN          NaN         NaN         NaN          ACTIVE  
1          0.0        0.0          0.0         0.0         NaN          ACTIVE  
2          0.0        0.0          0.0         0.0         1.0          ACTIVE  
3     

In [26]:
siesta2_actigraphy_combined_df = pd.concat(
    [df.assign(participant=name) for name, df in siesta2_dataframes.items()],
    ignore_index= True

)
## Verify
print(siesta2_actigraphy_combined_df.shape)
print(siesta2_actigraphy_combined_df['participant'].nunique(), 'participants')


(851846, 13)
34 participants


In [27]:
## Cleaning participant ID column names
siesta2_actigraphy_combined_df['subject_id'] = siesta2_actigraphy_combined_df['participant'].str[0:5]
siesta2_actigraphy_combined_df['subject_id'] = (siesta2_actigraphy_combined_df['subject_id'].astype(str).str.extract(r'(\d+)'))
siesta2_actigraphy_combined_df['subject_id'] = siesta2_actigraphy_combined_df['subject_id'].astype(int).apply(lambda x: f'siesta2_{x:03d}')
siesta2_actigraphy_combined_df['study'] = 'Siesta2'


siesta2_actigraphy_combined_df.head()

,Line,Date,Time,Off-Wrist Status,Activity,Marker,White Light,Red Light,Green Light,Blue Light,Sleep/Wake,Interval Status,participant,subject_id,study
0,1,2/7/2025,8:20:00 PM,0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,si275_2_7_2025_8_20_00_pm_si275_24hr_ah,siesta2_275,Siesta2
1,2,2/7/2025,8:21:00 PM,0,452.0,0.0,0.0,0.0,0.0,0.0,NaN,ACTIVE,si275_2_7_2025_8_20_00_pm_si275_24hr_ah,siesta2_275,Siesta2
2,3,2/7/2025,8:22:00 PM,0,706.0,0.0,0.0,0.0,0.0,0.0,1.0,ACTIVE,si275_2_7_2025_8_20_00_pm_si275_24hr_ah,siesta2_275,Siesta2
3,4,2/7/2025,8:23:00 PM,0,801.0,0.0,0.0,0.0,0.0,0.0,1.0,ACTIVE,si275_2_7_2025_8_20_00_pm_si275_24hr_ah,siesta2_275,Siesta2
4,5,2/7/2025,8:24:00 PM,0,801.0,0.0,0.0,0.0,0.0,0.0,1.0,ACTIVE,si275_2_7_2025_8_20_00_pm_si275_24hr_ah,siesta2_275,Siesta2


In [28]:
siesta2_actigraphy_combined_df['subject_id'].nunique()
## Seems good

34

### SNA Actigraphy

In [29]:
sna1_folder_path = '/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/SNA_Study_1_actigraphy'

In [30]:
## Step 3 Collect file metadata
sna1_files_to_import = []

for filename in os.listdir(sna1_folder_path):
    if filename.endswith('.csv'):
        filepath = os.path.join(sna1_folder_path, filename)
        sna1_files_to_import.append({
            'file': filename,
            'name': os.path.splitext(filename)[0].lower(),
            'filepath': filepath,
            'header_row': find_header_row(filepath),
            'delimiter': ',',
            'usecols': list(range(12))
        })

for file in sna1_files_to_import:
    print(file['name'], file['file'], '→ header row:', file['header_row'])

sna154844_11_12_2025_12_09_00_pm_sna154844_24hr_ks SNA154844_11_12_2025_12_09_00_PM_SNA154844_24hr_KS.csv → header row: 196
sna154813_2_23_2024_2_19_00_pm_sna154813_24hr_ks SNA154813_2_23_2024_2_19_00_PM_SNA154813_24hr_KS.csv → header row: 147
sna154816_3_20_2024_12_23_00_pm_sna154816_24hr_ks SNA154816_3_20_2024_12_23_00_PM_SNA154816_24hr_KS.csv → header row: 158
sna154821_3_28_2024_2_50_00_pm_sna154821_24hr_ks SNA154821_3_28_2024_2_50_00_PM_SNA154821_24hr_KS.csv → header row: 157
sna154842_10_24_2025_5_23_00_pm_sna154842_24hr_ks SNA154842_10_24_2025_5_23_00_PM_SNA154842_24hr_KS.csv → header row: 193
sna154815_3_1_2024_4_14_00_pm_sna154815_24hr_ks SNA154815_3_1_2024_4_14_00_PM_SNA154815_24hr_KS.csv → header row: 145
sna154835_3_28_2025_11_28_00_am_sna154835_24hr_ks SNA154835_3_28_2025_11_28_00_AM_SNA154835_24hr_KS.csv → header row: 191
sna154834_3_24_2025_3_48_00_pm_sna154834_24hr_ks SNA154834_3_24_2025_3_48_00_PM_SNA154834_24hr_KS.csv → header row: 146
sna154812_2_22_2024_4_21_00_pm_s

In [31]:
## Step 4 Import each file into a DataFrame
sna1_dataframes = {}
sna1_failed_files = []

for f in sna1_files_to_import:
    try:
        df = pd.read_csv(
            f['filepath'],
            header=f['header_row'],
            sep=f['delimiter'],
            usecols=f['usecols'],
            encoding='utf-8',
            encoding_errors='ignore',
            skip_blank_lines=True       # must match how find_header_row counts
        )

        ## Strip quotes from column names e.g. "Line" → Line
        df.columns = df.columns.str.strip('"').str.strip()

        ## Drop any fully blank rows
        df.dropna(how='all', inplace=True)

        ## Reset index so it runs 0, 1, 2... cleanly
        df.reset_index(drop=True, inplace=True)

        sna1_dataframes[f['name']] = df

    except Exception as e:
        print(f"FAILED: {f['file']} — {e}")
        sna1_failed_files.append(f['file'])

In [32]:
## Verify things uploaded properly
print(f"\n{len(sna1_dataframes)} files loaded successfully")
print(f"{len(sna1_failed_files)} files failed: {failed_files}")

## Spot-check one file
sample_key = list(sna1_dataframes.keys())[0]
print(f"\nSample: {sample_key}")
print(sna1_dataframes[sample_key].columns.tolist())
print(sna1_dataframes[sample_key].head())


37 files loaded successfully
0 files failed: []

Sample: sna154844_11_12_2025_12_09_00_pm_sna154844_24hr_ks
['Line', 'Date', 'Time', 'Off-Wrist Status', 'Activity', 'Marker', 'White Light', 'Red Light', 'Green Light', 'Blue Light', 'Sleep/Wake', 'Interval Status']
   Line        Date         Time  Off-Wrist Status  Activity  Marker  \
0     1  11/12/2025  12:09:00 PM                 0      36.0     0.0   
1     2  11/12/2025  12:10:00 PM                 0     121.0     1.0   
2     3  11/12/2025  12:11:00 PM                 0      36.0     0.0   
3     4  11/12/2025  12:12:00 PM                 0     208.0     0.0   
4     5  11/12/2025  12:13:00 PM                 0     298.0     0.0   

   White Light  Red Light  Green Light  Blue Light  Sleep/Wake Interval Status  
0       123.46       62.1         49.9        39.3         NaN          ACTIVE  
1       289.00      150.0        135.0       101.0         NaN          ACTIVE  
2        34.69       15.5         14.8        10.1        

In [33]:
sna1_actigraphy_combined_df = pd.concat(
    [df.assign(participant=name) for name, df in sna1_dataframes.items()],
    ignore_index= True

)
# Verify
print(sna1_actigraphy_combined_df.shape)
print(sna1_actigraphy_combined_df['participant'].nunique(), 'participants')

(517069, 13)
37 participants


In [34]:
sna1_actigraphy_combined_df['subject_id'] = sna1_actigraphy_combined_df['participant'].str[0:9].astype(str)
sna1_actigraphy_combined_df['subject_id'] = sna1_actigraphy_combined_df['subject_id'].str.extract(r'(\d+)')
sna1_actigraphy_combined_df['subject_id'] = sna1_actigraphy_combined_df['subject_id'].astype(int).apply(lambda x: f'SNA_{x:06d}')
sna1_actigraphy_combined_df['study'] = 'SNA1'
sna1_actigraphy_combined_df.head()

,Line,Date,Time,Off-Wrist Status,Activity,Marker,White Light,Red Light,Green Light,Blue Light,Sleep/Wake,Interval Status,participant,subject_id,study
0,1,11/12/2025,12:09:00 PM,0,36.0,0.0,123.46,62.1,49.9,39.3,NaN,ACTIVE,sna154844_11_12_2025_12_09_00_pm_sna154844_24h...,SNA_154844,SNA1
1,2,11/12/2025,12:10:00 PM,0,121.0,1.0,289.00,150.0,135.0,101.0,NaN,ACTIVE,sna154844_11_12_2025_12_09_00_pm_sna154844_24h...,SNA_154844,SNA1
2,3,11/12/2025,12:11:00 PM,0,36.0,0.0,34.69,15.5,14.8,10.1,1.0,ACTIVE,sna154844_11_12_2025_12_09_00_pm_sna154844_24h...,SNA_154844,SNA1
3,4,11/12/2025,12:12:00 PM,0,208.0,0.0,421.23,167.0,197.0,127.0,1.0,ACTIVE,sna154844_11_12_2025_12_09_00_pm_sna154844_24h...,SNA_154844,SNA1
4,5,11/12/2025,12:13:00 PM,0,298.0,0.0,76.39,24.2,29.4,18.1,1.0,ACTIVE,sna154844_11_12_2025_12_09_00_pm_sna154844_24h...,SNA_154844,SNA1


In [35]:
sna1_actigraphy_combined_df['subject_id'].unique()

array(['SNA_154844', 'SNA_154813', 'SNA_154816', 'SNA_154821',
       'SNA_154842', 'SNA_154815', 'SNA_154835', 'SNA_154834',
       'SNA_154812', 'SNA_154810', 'SNA_154843', 'SNA_154826',
       'SNA_154833', 'SNA_154825', 'SNA_154808', 'SNA_154847',
       'SNA_154801', 'SNA_154840', 'SNA_154838', 'SNA_154853',
       'SNA_154848', 'SNA_154845', 'SNA_154804', 'SNA_154814',
       'SNA_154818', 'SNA_154827', 'SNA_154830', 'SNA_154837',
       'SNA_154817', 'SNA_154829', 'SNA_154811', 'SNA_154809',
       'SNA_154828', 'SNA_154831', 'SNA_154836', 'SNA_154850',
       'SNA_154854'], dtype=object)

### SPU Actigraphy

In [36]:
spu1_folder_path = '/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/SPU_Study_1_actigraphy'

In [37]:
## Step 3 Collect file metadata
spu1_files_to_import = []

for filename in os.listdir(spu1_folder_path):
    if filename.endswith('.csv'):
        filepath = os.path.join(spu1_folder_path, filename)
        spu1_files_to_import.append({
            'file': filename,
            'name': os.path.splitext(filename)[0].lower(),
            'filepath': filepath,
            'header_row': find_header_row(filepath),
            'delimiter': ',',
            'usecols': list(range(12))
        })

for file in spu1_files_to_import:
    print(file['name'], file['file'], '→ header row:', file['header_row'])

spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks SPU154459_9_5_2023_11_03_00_AM_SPU154459_24hr_KS.csv → header row: 155
spu154301_10_27_2022_12_44_00_pm_spu154301_24hr_ks SPU154301_10_27_2022_12_44_00_PM_SPU154301_24hr_KS.csv → header row: 154
spu154433_3_22_2023_4_03_00_pm_spu154433_24hr_ks SPU154433_3_22_2023_4_03_00_PM_SPU154433_24hr_KS.csv → header row: 160
spu154413_2_8_2023_3_18_00_pm_spu154413_24hr_ks SPU154413_2_8_2023_3_18_00_PM_SPU154413_24hr_KS.csv → header row: 201
spu154402_1_24_2023_1_56_00_pm_spu154402_24hr_ks SPU154402_1_24_2023_1_56_00_PM_SPU154402_24hr_KS.csv → header row: 149
spu154457_9_8_2023_12_46_00_pm_spu154457_24hr_ks SPU154457_9_8_2023_12_46_00_PM_SPU154457_24hr_KS.csv → header row: 161
spu154409_2_1_2023_9_57_00_am_spu154409_24hr_ks SPU154409_2_1_2023_9_57_00_AM_SPU154409_24hr_KS.csv → header row: 214
spu154414_1_31_2023_12_47_00_pm_spu154414_24hr_ks SPU154414_1_31_2023_12_47_00_PM_SPU154414_24hr_KS.csv → header row: 157
spu154360_3_25_2022_1_51_00_pm_spu1543

In [38]:
## Step 4 Import each file into a DataFrame
spu1_dataframes = {}
spu1_failed_files = []

for f in spu1_files_to_import:
    try:
        df = pd.read_csv(
            f['filepath'],
            header=f['header_row'],
            sep=f['delimiter'],
            usecols=f['usecols'],
            encoding='utf-8',
            encoding_errors='ignore',
            skip_blank_lines=True       # must match how find_header_row counts
        )

        ## Strip quotes from column names e.g. "Line" → Line
        df.columns = df.columns.str.strip('"').str.strip()

        ## Drop any fully blank rows
        df.dropna(how='all', inplace=True)

        ## Reset index so it runs 0, 1, 2... cleanly
        df.reset_index(drop=True, inplace=True)

        spu1_dataframes[f['name']] = df

    except Exception as e:
        print(f"FAILED: {f['file']} — {e}")
        spu1_failed_files.append(f['file'])

In [39]:
## Verify things uploaded properly
print(f"\n{len(spu1_dataframes)} files loaded successfully")
print(f"{len(spu1_failed_files)} files failed: {failed_files}")

## Spot-check one file
sample_key = list(spu1_dataframes.keys())[0]
print(f"\nSample: {sample_key}")
print(spu1_dataframes[sample_key].columns.tolist())
print(spu1_dataframes[sample_key].head())


103 files loaded successfully
0 files failed: []

Sample: spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks
['Line', 'Date', 'Time', 'Off-Wrist Status', 'Activity', 'Marker', 'White Light', 'Red Light', 'Green Light', 'Blue Light', 'Sleep/Wake', 'Interval Status']
   Line      Date         Time  Off-Wrist Status  Activity  Marker  \
0     1  9/5/2023  11:03:00 AM                 0     159.0     1.0   
1     2  9/5/2023  11:04:00 AM                 0     308.0     0.0   
2     3  9/5/2023  11:05:00 AM                 0     134.0     0.0   
3     4  9/5/2023  11:06:00 AM                 0     129.0     0.0   
4     5  9/5/2023  11:07:00 AM                 0     482.0     0.0   

   White Light  Red Light  Green Light  Blue Light  Sleep/Wake Interval Status  
0       547.78      461.0        284.0       230.0         NaN          ACTIVE  
1       467.36      333.0        252.0       177.0         NaN          ACTIVE  
2       231.51      220.0        124.0       107.0         1.0         

In [40]:
spu1_actigraphy_combined_df = pd.concat(
    [df.assign(participant=name) for name, df in spu1_dataframes.items()],
    ignore_index= True

)
## Verify
print(spu1_actigraphy_combined_df.shape)
print(spu1_actigraphy_combined_df['participant'].nunique(), 'participants')
spu1_actigraphy_combined_df.head()

(1309582, 13)
103 participants


,Line,Date,Time,Off-Wrist Status,Activity,Marker,White Light,Red Light,Green Light,Blue Light,Sleep/Wake,Interval Status,participant
0,1,9/5/2023,11:03:00 AM,0,159.0,1.0,547.78,461.0,284.0,230.0,NaN,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks
1,2,9/5/2023,11:04:00 AM,0,308.0,0.0,467.36,333.0,252.0,177.0,NaN,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks
2,3,9/5/2023,11:05:00 AM,0,134.0,0.0,231.51,220.0,124.0,107.0,1.0,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks
3,4,9/5/2023,11:06:00 AM,0,129.0,0.0,1095.13,420.0,518.0,267.0,1.0,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks
4,5,9/5/2023,11:07:00 AM,0,482.0,0.0,191.45,70.9,70.5,42.3,1.0,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks


In [41]:
## Cleaning subject_id names
spu1_actigraphy_combined_df['subject_id'] = spu1_actigraphy_combined_df['participant'].str[0:9].astype(str)
spu1_actigraphy_combined_df['subject_id'] = spu1_actigraphy_combined_df['subject_id'].str.extract(r'(\d+)')
spu1_actigraphy_combined_df['subject_id'] = spu1_actigraphy_combined_df['subject_id'].astype(int).apply(lambda x: f'SPU_{x:06d}')
spu1_actigraphy_combined_df['study'] = 'SPU1'

spu1_actigraphy_combined_df.head()

,Line,Date,Time,Off-Wrist Status,Activity,Marker,White Light,Red Light,Green Light,Blue Light,Sleep/Wake,Interval Status,participant,subject_id,study
0,1,9/5/2023,11:03:00 AM,0,159.0,1.0,547.78,461.0,284.0,230.0,NaN,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks,SPU_154459,SPU1
1,2,9/5/2023,11:04:00 AM,0,308.0,0.0,467.36,333.0,252.0,177.0,NaN,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks,SPU_154459,SPU1
2,3,9/5/2023,11:05:00 AM,0,134.0,0.0,231.51,220.0,124.0,107.0,1.0,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks,SPU_154459,SPU1
3,4,9/5/2023,11:06:00 AM,0,129.0,0.0,1095.13,420.0,518.0,267.0,1.0,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks,SPU_154459,SPU1
4,5,9/5/2023,11:07:00 AM,0,482.0,0.0,191.45,70.9,70.5,42.3,1.0,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks,SPU_154459,SPU1


In [42]:
spu1_actigraphy_combined_df['subject_id'].unique()

array(['SPU_154459', 'SPU_154301', 'SPU_154433', 'SPU_154413',
       'SPU_154402', 'SPU_154457', 'SPU_154409', 'SPU_154414',
       'SPU_154360', 'SPU_154074', 'SPU_154335', 'SPU_154312',
       'SPU_154473', 'SPU_154304', 'SPU_154344', 'SPU_154468',
       'SPU_154478', 'SPU_154324', 'SPU_154441', 'SPU_154407',
       'SPU_154467', 'SPU_154348', 'SPU_154423', 'SPU_154340',
       'SPU_154303', 'SPU_154464', 'SPU_154449', 'SPU_154345',
       'SPU_154428', 'SPU_154419', 'SPU_154326', 'SPU_154352',
       'SPU_154509', 'SPU_154506', 'SPU_154421', 'SPU_154514',
       'SPU_154377', 'SPU_154427', 'SPU_154308', 'SPU_154451',
       'SPU_154404', 'SPU_154310', 'SPU_154426', 'SPU_154440',
       'SPU_154314', 'SPU_154364', 'SPU_154121', 'SPU_154401',
       'SPU_154417', 'SPU_154370', 'SPU_154025', 'SPU_154057',
       'SPU_154511', 'SPU_154049', 'SPU_154445', 'SPU_154138',
       'SPU_154332', 'SPU_154351', 'SPU_154090', 'SPU_154418',
       'SPU_154403', 'SPU_154122', 'SPU_154325', 'SPU_1

## Merging actigraphy Data

In [43]:
display(DXA_actigraphy_combined_df.head())
display(siesta2_actigraphy_combined_df.head())
display(sna1_actigraphy_combined_df.head())
display(spu1_actigraphy_combined_df.head())

,Line,Date,Time,Off-Wrist Status,Activity,Marker,White Light,Red Light,Green Light,Blue Light,Sleep/Wake,Interval Status,participant,subject_id,study
0,1.0,3/27/2025,12:49:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
1,2.0,3/27/2025,12:50:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
2,3.0,3/27/2025,12:51:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
3,4.0,3/27/2025,12:52:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
4,5.0,3/27/2025,12:53:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA


,Line,Date,Time,Off-Wrist Status,Activity,Marker,White Light,Red Light,Green Light,Blue Light,Sleep/Wake,Interval Status,participant,subject_id,study
0,1,2/7/2025,8:20:00 PM,0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,si275_2_7_2025_8_20_00_pm_si275_24hr_ah,siesta2_275,Siesta2
1,2,2/7/2025,8:21:00 PM,0,452.0,0.0,0.0,0.0,0.0,0.0,NaN,ACTIVE,si275_2_7_2025_8_20_00_pm_si275_24hr_ah,siesta2_275,Siesta2
2,3,2/7/2025,8:22:00 PM,0,706.0,0.0,0.0,0.0,0.0,0.0,1.0,ACTIVE,si275_2_7_2025_8_20_00_pm_si275_24hr_ah,siesta2_275,Siesta2
3,4,2/7/2025,8:23:00 PM,0,801.0,0.0,0.0,0.0,0.0,0.0,1.0,ACTIVE,si275_2_7_2025_8_20_00_pm_si275_24hr_ah,siesta2_275,Siesta2
4,5,2/7/2025,8:24:00 PM,0,801.0,0.0,0.0,0.0,0.0,0.0,1.0,ACTIVE,si275_2_7_2025_8_20_00_pm_si275_24hr_ah,siesta2_275,Siesta2


,Line,Date,Time,Off-Wrist Status,Activity,Marker,White Light,Red Light,Green Light,Blue Light,Sleep/Wake,Interval Status,participant,subject_id,study
0,1,11/12/2025,12:09:00 PM,0,36.0,0.0,123.46,62.1,49.9,39.3,NaN,ACTIVE,sna154844_11_12_2025_12_09_00_pm_sna154844_24h...,SNA_154844,SNA1
1,2,11/12/2025,12:10:00 PM,0,121.0,1.0,289.00,150.0,135.0,101.0,NaN,ACTIVE,sna154844_11_12_2025_12_09_00_pm_sna154844_24h...,SNA_154844,SNA1
2,3,11/12/2025,12:11:00 PM,0,36.0,0.0,34.69,15.5,14.8,10.1,1.0,ACTIVE,sna154844_11_12_2025_12_09_00_pm_sna154844_24h...,SNA_154844,SNA1
3,4,11/12/2025,12:12:00 PM,0,208.0,0.0,421.23,167.0,197.0,127.0,1.0,ACTIVE,sna154844_11_12_2025_12_09_00_pm_sna154844_24h...,SNA_154844,SNA1
4,5,11/12/2025,12:13:00 PM,0,298.0,0.0,76.39,24.2,29.4,18.1,1.0,ACTIVE,sna154844_11_12_2025_12_09_00_pm_sna154844_24h...,SNA_154844,SNA1


,Line,Date,Time,Off-Wrist Status,Activity,Marker,White Light,Red Light,Green Light,Blue Light,Sleep/Wake,Interval Status,participant,subject_id,study
0,1,9/5/2023,11:03:00 AM,0,159.0,1.0,547.78,461.0,284.0,230.0,NaN,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks,SPU_154459,SPU1
1,2,9/5/2023,11:04:00 AM,0,308.0,0.0,467.36,333.0,252.0,177.0,NaN,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks,SPU_154459,SPU1
2,3,9/5/2023,11:05:00 AM,0,134.0,0.0,231.51,220.0,124.0,107.0,1.0,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks,SPU_154459,SPU1
3,4,9/5/2023,11:06:00 AM,0,129.0,0.0,1095.13,420.0,518.0,267.0,1.0,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks,SPU_154459,SPU1
4,5,9/5/2023,11:07:00 AM,0,482.0,0.0,191.45,70.9,70.5,42.3,1.0,ACTIVE,spu154459_9_5_2023_11_03_00_am_spu154459_24hr_ks,SPU_154459,SPU1


In [44]:
full_actigraphy_df = pd.concat([DXA_actigraphy_combined_df, siesta2_actigraphy_combined_df, sna1_actigraphy_combined_df, spu1_actigraphy_combined_df], axis=0)

full_actigraphy_df.shape

(4476826, 15)

In [45]:
full_actigraphy_df.head()

,Line,Date,Time,Off-Wrist Status,Activity,Marker,White Light,Red Light,Green Light,Blue Light,Sleep/Wake,Interval Status,participant,subject_id,study
0,1.0,3/27/2025,12:49:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
1,2.0,3/27/2025,12:50:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
2,3.0,3/27/2025,12:51:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
3,4.0,3/27/2025,12:52:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA
4,5.0,3/27/2025,12:53:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA


# Setting up Time-series data

In [46]:
# Step 1: Create a DataFrame with one-minute intervals from 00:00 to 23:59
time_range = pd.date_range(start="00:00", end="23:59", freq="min").time  # 1T represents 1-minute frequency
df = pd.DataFrame({'time': time_range})

# Step 2: Create a dictionary that maps each time (HH:MM) to a unique integer (1 to 1440)
time_to_epoch = {time.strftime('%H:%M'): i+1 for i, time in enumerate(time_range)}

# Step 3: Create a new column 'epoch_num' based on the dictionary
df['time_str'] = df['time'].apply(lambda x: x.strftime('%H:%M'))  # Create a column with time in 'HH:MM' format
df['epoch_num'] = df['time_str'].map(time_to_epoch)  # Map each time to the corresponding epoch number

# Display the DataFrame
df.head()  # Print the first 10 rows to verify the output


,time,time_str,epoch_num
0,00:00:00,00:00,1
1,00:01:00,00:01,2
2,00:02:00,00:02,3
3,00:03:00,00:03,4
4,00:04:00,00:04,5


In [47]:
# Create the time-to-epoch mapping as shown earlier
time_range = pd.date_range(start="00:00", end="23:59", freq="min").time
time_to_epoch = {time.strftime('%H:%M'): i+1 for i, time in enumerate(time_range)}

# Function to convert 'HH:MM' format to 'epoch_num'
def convert_to_epoch(time_str):
    return time_to_epoch.get(time_str, None)  # Use .get() to avoid KeyError for missing values

# Sample function that handles both sleep_time and wake_time
def map_sleep_epochs(row):
    row['sleep_epoch'] = convert_to_epoch(row['sleep_time'])
    row['wake_epoch'] = convert_to_epoch(row['wake_time'])
    return row


In [48]:
## Prep time-series data, add date_time24 and time_epoch
full_actigraphy_df.columns = full_actigraphy_df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
full_actigraphy_df['date'] = full_actigraphy_df['date'].astype(str)
full_actigraphy_df['time'] = full_actigraphy_df['time'].astype(str)
full_actigraphy_df['date_time24'] = pd.to_datetime(full_actigraphy_df['date'] + ' ' + full_actigraphy_df['time'], errors='coerce')
full_actigraphy_df['time_epoch'] = full_actigraphy_df['date_time24'].dt.hour*60 + full_actigraphy_df['date_time24'].dt.minute


full_actigraphy_df.head()

,line,date,time,off_wrist_status,activity,marker,white_light,red_light,green_light,blue_light,sleep/wake,interval_status,participant,subject_id,study,date_time24,time_epoch
0,1.0,3/27/2025,12:49:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA,2025-03-27 12:49:00,769.0
1,2.0,3/27/2025,12:50:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA,2025-03-27 12:50:00,770.0
2,3.0,3/27/2025,12:51:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA,2025-03-27 12:51:00,771.0
3,4.0,3/27/2025,12:52:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA,2025-03-27 12:52:00,772.0
4,5.0,3/27/2025,12:53:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA,2025-03-27 12:53:00,773.0


In [49]:
full_actigraphy_df.dtypes

line                       float64
date                        object
time                        object
off_wrist_status           float64
activity                   float64
marker                     float64
white_light                float64
red_light                  float64
green_light                float64
blue_light                 float64
sleep/wake                 float64
interval_status             object
participant                 object
subject_id                  object
study                       object
date_time24         datetime64[ns]
time_epoch                 float64
dtype: object

In [50]:
full_actigraphy_df.drop(['line', 'white_light', 'red_light', 'green_light', 'blue_light'], axis=1, inplace=True)

full_actigraphy_df.head()

,date,time,off_wrist_status,activity,marker,sleep/wake,interval_status,participant,subject_id,study,date_time24,time_epoch
0,3/27/2025,12:49:00 PM,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA,2025-03-27 12:49:00,769.0
1,3/27/2025,12:50:00 PM,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA,2025-03-27 12:50:00,770.0
2,3/27/2025,12:51:00 PM,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA,2025-03-27 12:51:00,771.0
3,3/27/2025,12:52:00 PM,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA,2025-03-27 12:52:00,772.0
4,3/27/2025,12:53:00 PM,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah,DXA_174,DXA,2025-03-27 12:53:00,773.0


In [51]:
first_cols = ['subject_id', 'study', 'date', 'time', 'time_epoch', 'date_time24']
remaining_cols = [ col for col in full_actigraphy_df.columns if col not in first_cols]

full_actigraphy_df = full_actigraphy_df[first_cols + remaining_cols]

full_actigraphy_df.head()

,subject_id,study,date,time,time_epoch,date_time24,off_wrist_status,activity,marker,sleep/wake,interval_status,participant
0,DXA_174,DXA,3/27/2025,12:49:00 PM,769.0,2025-03-27 12:49:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
1,DXA_174,DXA,3/27/2025,12:50:00 PM,770.0,2025-03-27 12:50:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
2,DXA_174,DXA,3/27/2025,12:51:00 PM,771.0,2025-03-27 12:51:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
3,DXA_174,DXA,3/27/2025,12:52:00 PM,772.0,2025-03-27 12:52:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
4,DXA_174,DXA,3/27/2025,12:53:00 PM,773.0,2025-03-27 12:53:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah


In [52]:
## Export data for merging with scored (scrubbed data)

full_actigraphy_df.to_csv('full_actigraphy_df_cleaned.csv')